# ACE-Step: uji konteks dan penyambungan bass

Gunakan **VM Colab CUDA yang sudah punya checkout bridge/runtime/bobot dari notebook 10**. Jika perlu, salin sel-sel ini ke notebook pada VM yang masih aktif. Tool ZIP ini bukan installer model. Helper bridge sudah disertakan di notebook ini. **Upload piano asli saja** saat diminta; tidak perlu mengunduh atau mengunggah ZIP tools.

Semua flag inferensi awalnya **False**. Konteks **90 detik sudah dipilih pendengar**; 120 detik dilaporkan banyak noise. Jalankan sel persiapan lalu bagian chunk seluruh lagu. Probe durasi bersifat opsional dan boleh dilewati. Kualitas sambungan lagu penuh belum divalidasi.


In [ ]:
from pathlib import Path
import base64, gzip, hashlib, importlib.util, json, sys, tempfile, uuid, zipfile
from google.colab import files

for dependency in ("numpy", "scipy", "soundfile"):
    assert importlib.util.find_spec(dependency), f"Kernel notebook memerlukan {dependency}; jangan ubah environment model pinned"

# Generated from repository scripts by scripts/build_context_bridge.py.
# BEGIN EMBEDDED CONTEXT HELPERS
EMBEDDED_HELPERS = {
  "ace_context_chunks.py": {
    "sha256": "8ab7629c197c5f557ba20b1e978791db23fe5ef16044f76a5eac0e227b9610c2",
    "gzip_base64": "H4sIAAAAAAAC/60825LbRnbv8xWQ9gHADAhyKEtrcwaq8irazVYSW2U7qVQxLFQTaJKIQIAGwLlYUVX+IX+YL8m5NdAAQUouRw8jsi+n+5w+9z7Nly9ffv/u/eTnRh+cdXksUp1OkrJo9FPj6KeDrrK9LpraUUXq7LWqj5VOA6dW+0OuJyrPtoVOnbrJmmSXFdvw6uonfax17TQ77SQ7nXyE7rWq62laHffOY1l91FUdOh9yVUwPVbnWTqo36pg3TlM6RelkxUZXukh0ePWD1mntFMf94RlWTDL6D7e4yXIdOJvN/qC3gXPIj7ycfspgI8XWefev//C9Ux2LBvbubMrKmUz0k06ODQB9+fLlVbY/lFXjqGp7UFWtzfedqnd5tjZf/7MuC/N5r5rd1aYq984BPsEgRzo+YId8rnfHJsvbb8c14Jfour4yTYSKowCnQzvK4IPN9YbXIFzDGmircrNQUlaVzlWDiGdFGh+0+thB1rlOmipL4s0xz2PY+JbgATFyDUfy/S/vnch5NZ/NZlc//fjjL/AFN+7FMS4dx35Y6brMH7Tnh0ASPPDl7erq6grOBtBS89dvPETcX1w58C/NtrpuIqFXKAN86nvMmh0DpwlhedCF51Zr16cdNZVWe4aC//Bw1nmZfIRzd7JGV16u9utULXggbEul3u1s/s01/vGDtev6C14+PB5SIIdH03ntSjfHqpDthTv9xJ9gZ4zJYwVLxHiutLngQeVHLThhQ9Tt+67RSFlVPUf4PUS04vq42WRPnhs2+4PLS7bDQgaOcuPhCmEKh117tEQABwY0jeaByvPyMS5UEf1V5bX2b9z/KE4gVfqQq0TzRmTruXouj43XlI3KA0cENPpuFgBrVtusiN4AX6hUR+Hrb6H/qOvoh7IAZtGFrrbP9EUwpS3VkSdAAgYQ4GzeSbZxPFU8exnwT90okEXvIViXZe47cFxF2Ti9Hi8rmmCTl6rx2wEoL2FWA6cCUbwHnw76AQ+ZV/dbDpAJt7P7SDZ0H72ZzUz77B73dc97xEb+dH+Ln2XCZH5tWmd+x1uVymrt/Buu976qyspzUaGYSbDi//73/8BKtSHh2+i2ZiLiPr2ZEMZHclcaeme12xJoQAU+FyCE3yF0jWJ3H1EX4UQNl/b3c3msEu3sj3XjgGI0O3RqDZtOzep/cv4OGGw10KJMde6kx0o1WVnUsO0kP8L2Nejqg0pTVMjOD/oBhuqnBJFHNVnpX+EEGiSFOoQEcq+esv1xH9GxwVHCZoRMPm1aJCSNKlRXHhOGevw7ok4kEOAoYJghEvbc334Zb4OPORrYl5Nr9QBmJAf9CbtWBVCjdkqG6ahjmpVCjgS21ES3uJ6QWvYCShHWgS/ePCDEEp3lHh8V79OfIjRf4KiDSrImA9FYYvMNDECgGXLDchbQOpPbFYOlbSBTU3eliq32aIS/6rDHjUW3Cxuy7I0HgUWCptnqzjmUeZbQANNDnwkMfARWc1CCZW1oMYsM5LQ5K6fNV8hp04llJ5mz++aeaDYlM4I4N4gz7oLg4IcXUQ1mSKderRuPei4K4l+QjUDVOWih65bheRuBcywyYFE09giTXA/AIwPORv5F6yZnPziAWz4ke2m9VwC02LIYTpDiSzjEdkADrKybyLTfMIO30667s1tm0D+tj3uv37ZY+R3Z8hK5z2vBCfMHvPpwcm/qLov2WdFNHawsIJjJJpl/LaA7AMANefl2ly1GFF/pbMB3y9bgZ6wN7dHXq4YagSVwKvo2z/ZZY/SORS7c6C4LENW8DLixR4UAsbF7QeBot4FM5dab+QCH8kCaLFryOTTc3Wc6RvQ+6g25j3bZyiaFQFrYHI0CR3uXzuCjfo7Y4XCahVrXXjMxyIhMPkcueHEPwHtpbCjntjBRHAcrALNm6JfUUQ7eqMeMSQS5uQ1wq9MpWCkSnOU4GVejW+4A93ftsYEH2UWs+QuriZl/E85m8+seXtM+vUmvC57omfD8uNCqinmGOwTrrlWOGiWNUXg7WiDjhuoA7l6KTOx3Ss208kL+1XA0cbY0747Fx1YLkngHngrWPh6+Bv9ZV+jz/ZYdSFACkhYSwe4YQOFVTQCAiQNngZqgqifGYzO9RtXe0cDYzwhImU5oNpOpHQHhi5YggI2ImeK3TktHBsLAoPbJRdfvyV1kgYsmIybw7kLJVxjjLtYBjDocG9PJCEgbjYA/Qe/MrH8ueQGx2ZK7GNnm555//Mnl8C1GWroLUlAukSbeVAoUsrtgQrmCXAy6KDZOyML4jVf9XZDG6EaJV+miP9W14rfANYIUGw5xF+YTrEkEhGXo/8/i/2IgEJPZ7wUiqlFBXUX1hiMF8upV/qie63ieRr9UYEXS5vmgI5cM36t558HV1YuILBowGcLBMOagIex5Ec2N7SsOnXHEMX4IDrz3ZR+TpwAnQihbQtz18R9/6/ksJlABkIIf2mkIkCFKiMuNB/5/Iat0HRG2hiCSnts1ukHXeoAgDgQ2+PTZPx3mYhDu+i36XR97CuDx8pDAxUgdho7YkZ+PhwPbZDodC0i9oCg/cGSyjWY3ysRhlAWIQby9rs/ga28tinhPC4EkAe3YOF63OxkJjLE5pjyD5RMw8fs9fwhZRgtj6Zqwss5PVvPcbr0Q40ND5/BRPZjPFL27fh+3lh9aJFkPey5TIz7AVgzINg+A+xPQvbZ2s8AsGOl7NcUcgQPRJaibrwstKWsg8eV1YJ+DsBBgLVkcllk7/OTDj8Z5wBztYD76v8il5AOLOyhTXrRHP3KCO8Aw7YAhmLo9VbQuNNUpi/xZWFbIwdTgfAA3WSmSgJp5iNVM87k13H9MM2ADTqWwHqLsVFx+lMifR1cZEBj0LqZSIsmjyHo0YIuWl3R4TOYgYvhTj62De8ODQ85MhGDCdeWJkHM6KkzKwzOelznpIchOIfLyJ/0vot42Lym/H2WgQzNBbtD7SZ3HHSa4mOMwP4c+J7K0ieAg1isMZq5ghLwr4e47VZRFlgDYlhnw7BAC+ujTffaEUPEUQ+dvvH0IJjCTRiuhWqwedCrgDDZw1rQnTtIhBLCcyU5yl+i9IagJWHJwADHJSeI2QXFzSI5rDp67RF9YHQtv6XJiEiRvUpR1A1E4fiRJRCpRO/7J3ACY2Bw3NKiKW9hRHrX3MCiBuXOcnywU/HdI9nF9+ybXPJdo6a8CSr0S4zGNWQwtK8oDu2ATRauNXI2Md2e9141CcxVRbgtsaVp7JAbtWJ9MMCfA/F5UYuayUWJ8Y+Y2F7irz/Z9d/pC6GigOmmpaxPMJjs6POuIKaOCfGtFMYRwf1dp+VistYJgJ1iu/JvBlkGBYrxAfX0oJ2Ev+yXa+MweeVQWVpSujOgMluyf0ogJhwbf+gv0U3NdeDTE594b07saWjFyazzwU+pfK/p/D8fKOdHr67kv+0FRiSSFaMEeyQAGiEXAe+/mmlxrDWy1V/GDrmp0mOfAcqo51hAjoSzq1A1aVcGn2lcdgZw9mQ9b0E+ZvcclhkWYZwP2c8SChB9+ev/z+18Cywh1H0/hWtrN3sxQ6YUFUD04HTyK1ckiwIyA3w9ymYEmCPhJV9OWyULnnZXqkrRdLfa2FgvHObFeLu/vEExgXIFeG/vHd2Sws+KYNc98R3OsSVX+elQ5th0LOK1sk+k0dL/GvNq5k74NJ3EfNA7NH0VtVfmbLjpLVZMVjtuZ7s0AyKj1su4Z5PSHS98NDdxgQMAb6akiASU9LyJe4WtcB2PLIKZCU4GLWhoF+X95iukq+uTi1iDwoRWZrVzRfXKR8Zndou5uwlhCkjx27MgBtP1JbBBP7oRLvU2Zp7oKLC+UbjekfdpFDOPSwMstXbsJIuzBUePlYAK6L7oETjDtA5TGVS+ZfuBMZEyXXNmDjptS9tutS1kT+zrqRWR2cckr+Vu7LWMPzGECOF2DCIrNB/Pdj1xwke4WrHe2I0SGA9xb1Djhhi72goEmpWtCG9v+jjj/v8vNxXxeBlTrBWM9FsAdLF052cHB/rEDYcjmiC/edgy3jRCmqFHN+Zw9DPQ/YqSLnQwY3OB1u7wj82c5LoTaibcCeFuHR4YN3ZPWto1QHm+xxa9l/sroEqbE7EqjeyEFXyNyRDAdkS+b9GIZuzvZwTFYoOwTEA/qRTQqbZeOovOwv1ZAhFhLk3RYvRiP6FoG9sVQ037Jkw452TTcdi8RdXHXH2hpAsi7NvkuMBwbYKqaYwAr0jhVlRYpLSV7IuzjwyzGtDvFzRFeJfP7HJO5tpVGQC1tlgCDUmIF8Kn0nj/F0SClYI21+dlItQVlxBtHuJGMbNe4ax17nmwSRxhcYiKpp+qRIBcV3J1KGvA7aJwNb2CZOx6yIS9sQ41i0AHDM+6+tatLAGED6VQPzrH3AKyCxSG1lR9Dsd3bDSf7fBENl8XDdr8YpbyjjFE91HD7rKYgRQigc7zB67Acd8NOgP9k4JnMVAIKqqQbtAqLb9B5BF8QXL+qG2lJAd7v9yjT6ToRENB2Q/KJnLdiauTevsjvR02DUMBM/QOuwugynDQH4ESLZS+Vvurn8U/mthdfXfocAHk9SJiAX03GgPtfgC44bFVGYMM/nw4Z5O47LAYdPTIP1CeI8hD783vuwRGzwQD6HAHqwRzbxXQ3MzplSAq8FZoaMZuavbNFH3C+yefCOibBfyxYR9ZeVZasSGPLutuaUyyq5QW0g7+YYuwM0p2tdOMzylaWuuxjX/XCpeiiq3hlJac3SdHkXdWUUeNu2C0XYiTvmjIqdcNlVNhoFVEhGCzbSD56+Cfghn/+8d0/xe///b+sbz/8xQpZqnILMizJXLO0aeW442Rwz4uyAfTMDXkHvV5Kftaez6nrT597BWAmvS4ehVwArfo6Fi89KcFlmBqv1Vbg25HdNfvf8Oz4Ew97ScNerhazV+lnt3/rSfMkUdvmZruEmWWLYGnanWC0OIn3E5XsIBBq4ysZuISJq7EIeWDnePqpk0f+3WLUXcMJ5He07hkh7J6u0XM/eCXb/TgZf4DwtkEbA4bEMakDgY43z8EmP9Y7JtSd5B50D8qfnJ90UmLRkaKLsBr3qhqsbwObvmmwo6D8RlUdD5iJNwTjIDjs33+LM512GUqbdwzcrJBqEaFbuM3LNQgMd8fXPTXWZTjIk5JBtvs0dlSiIDNmaeBlTLdY8iD9tiD07Wtk29fFaH63d1qyL/u47jpySO/dGhb8OGTZdlgbZp5m2NiKsKE5PDe7suCMLqpfkGE4x43KcvAqNljrlMAyVFwLZ4frPhvSn2a/jvs1aN/ZSQdfA8gBgaiaw/nEM1hI/VZXLLj5Jro9FbiWCpeBnW6Bc4bmGNjUyMVICzQgW3nenxhPzXdlBL/LfQikMpkviUYhX197n05c00Xb8LmX0etdlYm29b+kFlrMz2mGnkpjPWzmhKcJArCtdqmtbQkC860fQUmj+ALp2rNLc3up7vnsujiAxG1vZx7mz7mVhwe3enJLSW/xKfa1pLpHAA1z5vT9iFeinD3kugFopeFvvvE7uAesdu9B3pdFyUl9BqaAh6Nbf7lYfLsKVY2gPAuUccMxF49T/fs335DDDRvm74DJN4u2XmP3m7ug61YXC/LLNEuy5tldzMLP7eqTCP/y6nLlk0Rt0ThBDeRPiuUQeNMcgDO6K1P4tgEPetnuBovaBMQUKSBFi95t0A2BZiXFTaZN2CwvH4Ndtt1JgdY3s9lsOn894wocq+3Va5/gqQQWlBwVFrWTJ2Yq3KF3CQAXCBCtPbbdgOf3aOdLEAr1+BwoLmer+2j21eRbY5KZmQLmEqAV4gwK/C5X2wgNIu79wNeO5AvgIKq/hAmrtxGCuA6/e21qSctik6XkDLbMluTZgbBR29WU9hjMglt/UJmDWyXqwDBW5AbS2yh8/ZolegSXbpypmKHCm0cNdANlatdFKtT7eKw5eMUHLDSfBfDtkE3nXD95yvr2HhGVEs4FoQCDB7iZFQIAgzJoNalTrfbxIxxp+QjMsmmCCjcV5KRngor/SzQ6BMFjlrYuv6hh6phQx3Q6v8PCLuq5oSZGKUK4S2qe5GPKFkuoxnuY09cRbUpAVGdBVOdBIDdwERGEkLS3YE4MuT5pHUkl/hXrzsFrgEVq4MjnEpyLdQkOnlzSmTTuowoe11HvcJlmd+scSaOuH9XN+vpxzRkGFWhADfWgH+B/awx7NlzoTerP0+paqxu9vtZrfzo3/FtVHeeyRhtRY9cwg+av2edHIaEV/begv/7cMatU2IB5jEC30z5wsz64XBtfzJc1ZRbKVf0HMmvOr8dMU2mKaLPUaUD11BkXcYIwFirPftNCLwxD85wTgeiirY9Znh6l0h1Yt4k8QhjLI7k6knbm33j43bud4Dq+1K9Xp/Z0NghhpdowIJSCTy4zrFUvR9/Z2LvIpzEQIE7XG+gDamhl3fW7xIWDAWufqgXZzmKFH34JXFqvCxl4DmapYZAF8QCxaEbpsVw/gKvHqV0aSsCnqOng1MR24lJ14y749PGzbzQKv/KKVVWp55pi5IBMXi3yKsWb/SjujrhTbAF39UoCV1YFsFhEnuq/iNqr6npMYt6DD/MswaOpIKesVNG4djEOKrLfdAU6yxvLMIM4DhTeq3lXXc7KSW5JC4cKT2l7fdSN/PN1pkj7V/qBoiWG9YZyvzooOLyUheFd0urTAu0BFSEOMi+kLoO18W6totRVYLfRfrtsEVFyqRbrldQuqFFcFuvJeQWJdqBf22uX7lsnb+Hb2QvesRTDm8+ru6LcY5JR1HcPn64iWCvQ2+bxCiip2/C15BkHZaur6St/UJJuVXR7S1lt1d1/nNayrmiL0cXCcfsfKRWmgoCf8IYD+XrDX9v68fnMcugx2m4UJnzRDwId7exUvuHnl1JjYB5PHZv2DQW9coIZWiU7R6dbK+huqmzfE9chiYg+xlIMKLRMWGqoVr9tJl3aWvC35qx6LHKD647SB3V4csNWrgXi37PJ70kXglhdnWRXzDbGH0YgV3L+jcoQ4AiLhkjipFmdqCq1BIc8RWQgz/ZpWBMwY5rPq8DydBJxa8Zo49vl/E+Lp+Wr1ZJV8KonsGJeMJdQk7tpccDfwfRiiVogDoScPliMROpE3BrO9UhPPCjxkuGDipLGFfqR2CEcCnufGRdKJJ8RHfRO7MMwukCNtZ7olPViwOZEq26x1XI9GWOY0WmjI8fVGNHTOlnMcvQwNK8HRpCQaq1X/pdwpHTXl1EZb5aqr1c9lr5UGNw3E/khqg7R2eiHm7BUyV24H9Cy26VE4vUoKiGYbKsS+Mu8/3Y/9x/A5IcAFuKgmOnoBxIiI/qWoqBq2Nv5bHZNDihE8XOvOixxi6tpLh/I9TJfSPrNkIFDKTFCU5vyNMr2RpnRm0ZlRfKdXTD2PpDC7AJEipvQmeKG9Zm3FuS7EWIRokyeGn+tDoywca4YUfp7BtZeFXzr95Dpx8gjNJZnXLqVc+9MXlFou67N0PNu3cp/++bcY5HeP1RG9LZ/WF2Ny1AP+PEYuJ9BgSvNBIdYik7SC4ksKYz7mR+2byDErZ20pLU1ZnCwbJdZbdpk+3WlQ+c91rFNDlgZ1hV6bo+Yemg0+Hs7VWuLb0NLWZO7YR7iENW6JIMV3ICUYy4DUJaKbjGueG0XsbMQQGQ/xWl8swEfmBNvQzsS4OnXOA+iAFrdXdB/gYvZaCwX1CoF/PZ8J7igoXiS9J0SHu4C/5597OMYgRUappnaYpFxlnAV+x3+hII6NiXIV5Y4Sa6yPb2a3ZUVnGuSHCuVPAcOkxcJV1Odc/JMd5P92sHQHXj+WN78R+7l+hdudMmIBXGjF29jN07e+C3VSK0Da1zwNe/MFYbtd37xwgkQPb3NGbl9Gr3dEb8DYIyXT12+pBqLdOSS1b7u6Spx2tKOXkoX+keTuUwYIxVWPTjdAvstucwIAsR9dkhlDoIZQ6euid/k/UObwx8p+5IpMpR/taO+a8v0nawxdw4K6LgDwedVXOtoA7m6OR+Snr7GkAvdDf9Sg6my3LifWkb8zMXIvAIZi/q45udjH979Szz/xh2H4e6zJ3q7w/PD19de+Odrm7hc4nQjddcXQCN71lQeWDjLdmNBu4LNpX/w4UGfAIy7374ryLP1/vAK/FKNU9bUdvvd/OP4XBjqjrw6IGzQv8Y4j05sKYrRQoPzIxJyYO9ymEJZDUKy0zMkd/wTTe6Jk3Wg4s9JOvEVgxxW23PvjfSuLp1Ue/gszIxeDcrI5rYuhdAJkskj9JVPYCRv+JJVXrF6IwH7hWsl6zG2LrbNjizZt2EfTC8nYQMewhgpTu5jjZwwos2kerlnzT7xVHngGdi1JrGhAdhNqzZlYA3RYJLHgT8EIJ5cl2w7RWbszq6DYSYymQK7MImRcxe2BYJTPF9m99n/w7KJr+foVRA9Q7ZeCZ0Swp003Mdb/92ie94Ixax/Y9au54Rb7rvN0xB5CdJahPEXGW3QEugnpiKl4SJLBkd8n7ZcMRpR1xYDmacCZ3W7WLERfmZvkfmVMeuleblJ3CH8cQiTlDtE5mekwu+rLeH4Ab9VXgq2usrolwaiOE7LJI79O+CLCLzNNI2RQ2hgjSObCHh2D1FBistbnrT1gAdm0Eye5rlc0kfVzdSuZH3PnUzY4rgBKS681h9AHZ9VYtXnuSnda6DhNPM7PEaYGAS52IH8ulf03ezMmpxeujj3zYW1MXa7ODl8/e2ZlfGpk4XuhUXsh9TgSmcJprr6z6VX7YLy0vo8tPaufsJX+cMtoNFAliUfwKWnhrAMenyBiFfPCxjyBU4FRjtZltjl8uka9xUgvDCiDLr4FIOy7DGKQR1/W2xsaalxAEtMD+kQNOoXoKl2T1bmzCngPLJwWxDJi+Mv2rTn8WYWfDcLbuez1ZDXT+GwaRhjF7ndgcE1CCqRFGfW3eMI/BaKsEb8PMC1X03VCMY8r6bBUmBCn0HEAoHAv60gvMuN/Vwrt9nXJTIV3+hZ6pUarVdvw5db1D98PmVVIA8QYuJbJXoXIjya20VtXWVdV5FH8OJPNNKcH8c6d19ThMdJna6a6GwVEQ08KSKiVruG6PfUaVoH+qXXDW1JERMb64ouF40GFzGQGiRmH7pibOuLeoRsy4eoVb70Tej/V/nQGK+gUurfgFnbWAjprNLejjgjjEPJxDjgWUF8nstooAC3UhODUVzJaP0WnwA2P8ZH5Tyw5Zj8gzgGdOIYTXsc48+XoIm/+j8MnFZZHlMAAA=="
  },
  "electric_full_song.py": {
    "sha256": "04d99be5e6bf1635b7ac16347680f344be13344fa1b8953523c9543a96c502ce",
    "gzip_base64": "H4sIAAAAAAAC/70863Lbxnr/9RRbnB8BFQqk7DpnyoSZUWQpVo8taSw77VTVYEBgSW4EAjgASJlWNdN36Bv2SfpddoHFhZTipvVkYmLv+91va8dxblSyiOVRFhSFOB7/93/+1w/jcSHOYhmWuQrFL9A+FCoJ43UEI8W6kCIoRCDiNJFincVpEMlInKZxMBNLGWcy9w4OPheyEOVSCvlFFSXOy1SSwLh8nZRqBUskkZDJQsEaJ9cXQ/GgymW6LoUqCxGpYJGkMC0Up+8vvisOjsej13CmMMg8cQUzinSdh1KEaVLKL+VQ4ElWaSRjGBLHP4okhSGwZ7hcJ/e4eZoLWK4Ml/DhHTiOc6BWWZqXIsgXWZAX0nyHabY1v3+HNczvVVAuze+0OJjn6Upk0BarmdDN19aQYj3L8jSURVG1bKufeH9eodxmACbdfAN/x/IyWMkiC8LqROu1iszvh2BTtX9V2VzF8uDg48mnMzEVr1+Nx+OD649nN2ef4PPxQMAfpwiXchX4G5kXKk2ciTgeCkdF8MORGsNHM8CwA80EQeyB3YtSZkeb4zfYKZ0hLxbJjQoljgjXUYBTslyGSq/szIEUytevsF0lRZmvVzIpscNsUEgZFdBwO77TKy7gckESSr8AxOHKf/XGNH0uc0ntcA6c8gab0zluEfll6ofZGlo/5WupVwLaKPU5LgqgxRJobQ7YhouX2xjo0NAzHkbAvmWQiyAM01UWJApP6okTEeXboUiCcp0HscjidXgPy5RIXg5tY/1xkGJFGMsgj7dAR0Cta942KMsgvAemKZaIqCQtpchh/wB5Auk+AE6axcg8wCxiGeSrNFFh4fVsciIKIguxWhcKYETHj5FrVAI8mKT5Sjce6TvlcgEcB+vO0zhOH5D6kQ8zFSTpd0XPFrw/3BvoEkfjCaNtEqzwSID6p4ODg0jOgSuLMAVC8vM0LV2ki6EA/AOmchlNLwFIgwnjwnHOFV5TzHIVLYBPlzK8R+Y2TI4gmaXpvVjE6SyIC+RPoKQ1EOlGAjSBGpIEz2KYGlaAszh8eNwaKBz5jY4x8HIJKN9Id0D9qyC/B7TDENcBETFXi2IEAJJlQdTsIV8jOZZpGhcjgNXKl5sgHoFo8rItEWqYq6yEPlmuM1+LLOzjDRAaMEFFLooAfWv8k8P4HPASxy51iREQ0woOqAofmdUdAFZyakP86YPyompeQ1OoAmEkEKj16rSivnc1dODJL0DBEYjl3G1DQq+LS/Wel84cKADnR77jWZ6nuTt3Pl5dfRLbADAQqUzFailm6/sgqTHJmJ2IR1zxyRm0QYDN1AaYjIDHS4n4AHjqkyEYgEAjoFMARKTC0kOJeC+3hatpi3AFwhlYc2CdGWcSJGCey0sMzVIjQB2pBsduugEMF7/KROYBaqKjD0hUhRZqnT+HPNFD0nQdVCOrIPNR4PmHI736YNgaBhLT19rI5zH+oTNoQVo1qcYCjRdE3Frjj8GUyyDaAuBu6cb21evJzWvAJob0nI1MojQfnZyeHd2gPD/23ow8aNyMZioZZVtgxsSxiLO5EsqBvUsRMWSpSoBT2ipjRJzrFcFcljIp0rywN7pj0limoE6ILkBOysjl28Il68tVvBHLxNUTBuJncWwxXZeAnZMIhI+cAcazoE20nrgolEAK999dXH4Cbk4WQNpFEAdL+H+5NjBWk660tP98L1BWeL8DBFygERdU3tBcalAhkNhBt4Liw7uYiwMVSuJxLWEJZj7jxUUpqwmIWwBO2LYXsQ70us6NFl88DwSEdHDbtPBI8kynwgHFzNs7NikcWAJDT65wth/g+ksL7JmM1ysRABIy/N9vHxCanvhnAHGCYqSAMSRbK3MQrAAxS2ew716QO5GqlEcxOh4D04HZ6Rvl7s/Xcewjz3oq2yazoYiDeC1AL4OlKRYkAgrlOQ3c8EU1Ckh3L23gg+HmIUHcTgB5wImAZdfgoaVBUC7Y3VqNOEDvvPo8R+vOlsJkRKBZ56UZUDjOpl4grHzmDMjSBmshndjy3KUmbyFLWhDEGuBnKKrWJFyC/gT0NlqLYJU9qAiUx2Ag/gGUIxqOQ/EK/utVCb8B6Ax6z75kAGFQTGhZyBSMzft3X8X16YfjH7qyvz6Hvu9AXx/YGg0dkIFfpQ/uw8JlWQmmB9jhskTogSE1BVsvWqOwTpPpMXwsApVMx95fh6Axv/hVF/gptcUBJ4xVqEp0SxZHWhaD98E2dAFHTxOwLGZbsIZAuz6kqHp/BLGyCMKtiMH0AccDtt8CqQIJlJW1QYY6EAEimfBt7PVc/n2tckmNNHKDACOzg65RX4Jv0Dx9xWtukGxdsKHBYkZL2N2A5oMVBygHkQ+bXS6cDnQbWtqDagg6J8CoYO4qoIQNGxkb1BJ8pKZgJ18Izid+EmOzwhg+zMnET9PGSfEbQG0Pxetg83FHKthEAz6g4DPxhj/biP1pau/xEyKTNA4tjap9PDy+07RVe1MeyAv31oa868znq0wukPecI/QawUtFuXy0wf9LPAp9cqNymEWZ8Ab9NoBzRP4KjcOD09KlbqkQiI3BHC0NZ5OCvJPTRzz8k7Nr0SDXSyDj8fQQD0Ue01E4YY8qXPnF8Q+x1GOZMwZ3Q1ZjU3R5GCpApGuw/adGquiRIHhwfUNewaxw9cijCvikQfXAPRicO29lmKJ3X9HCShVAbeFyQuSPOj8Sj6b3CQgdKOSR93tqillu1JIAzFe07Cr+B+2crXv53/Ir2OnOwQdJkB0MHxbL4NWbH4biIQdK88lp/2N8C1CymEzz7i4e1N0v4EMmnQa7PcctQZtfapVcYcByDdhEsw5XC5xdx69HNG6wT+dWUqN7xZqedPfxGMVCS3I8KyQayCbhhR7rwzKNUYDESB95DRsTowILAsR8VGgYpblaqCSI/WVQoJvEZGEYncGYzFPoQRLxMJBQuJZkIfbymRC7IgZGzSTLmJZcASf/wQcfJVdyp0/hzNFPL6fmojQxRdnhaGfUkknA6qi7mNM1q28CBXZOjG4vYczFq9zqZZ27W6da+W4AfE7kYxtzTbRVyxHe6sV/QvxBi2sTW0Vo9bifEave+M1exN6QUdbBlkjnbCD8SEhGg41wDEcE7Qw7wWFBiONJVipZgwvwAh6oj/a9wINZ2mzvETmMCKtRpEYi3YEdiueqpZv8Eso8Kw2VEX14q/tIoaWGzmlBmBpypNNP76fnARjWPBzDQ2i48zS0CQ2ZemD0sYERpqsVXgJsyz9RsXWV2N0+ME6qY3wP5+jRdneNs/Ig0n5G+YF9Rgf5Ji1HcNJ7tLS93rKr/8iXrtUfL2ETPZDQIak4lEM8Wpsyh89pvkrxEali8K1E4twhevaSJuhf3pvVbUsLI7W+5ByaCju0WtGpPgq7kIbXwBTlvb83ex+DfWAZCOCeUVB/2pp4aMbTdEYLhzV0EKomaG6vybnHo2HUWC6NYYJha6C1Bw5/0MPxlDKyA2H47YEHCRwYrArXfUV+DDs0GCm+vLo8QzpDVGCINwdqQhjViu5hiUJnFqfhvZhMqxN5GIPQFEUEPDG7kXnBPXnw4NLMRpRNw3LnBOffxw4A1jUwPxT/qM+DMFdIEYyKRx21/60RtNeYhq/HWtpPmnjD0CXyOrTT3087TNGZDEoKxIOucaL0IWk0hEBgkf7qn/8Q5HhWGuOcIIsckXdlX4SyMCoBYw2lKMeWR/dyO6LlxUqWQRSUgefcPTHdVOabWxOXvaCJ29ptg91TtYkZmWmP/bmQnaCsbl4foiDgn2COwaL6p6E1xGezoxrYsEKYsJ/6loa71FP1pH1gGDxpftdgJLKpTKBqpYZRhOdETuxspPmzg2t9y874FptiYCGWPoYhYAwzoUOCw2fyh1b67NnBAr8tJU02yt9D6TKJfM1L9T66oe8uNmPU/CQje4/aPDZ3R5cOOkC77SFTlaBRoAnNoKTh/ZhG7f/ILzIE28bnMAQFjGon6JDyKiDdyB7G3KtJpk1NKo2i/5QgIwXVyr58lJikRTMGpurAjc61gtkC3/GWE65CK9ojEGYB5lkohzY6v379aqRDJ5jxyVVkpV/4bJiI4FwjJy6wyViLFFjktsrBqu7AHgk4+FVaMIIOUKVoefIkjB25dg6R4lX19z6F+S8EUXs/Ut7ktLYsO95MK0478LfP22Q3YZ3B8jJY9XudOqgJXsFG5WlCpzDOZ7eL5tlDp32j7JNZKSTXeXfuv/v8i396cvqOVB58/+vZp/r7w/X7X05O/3Z2+VZ/nV5dnl/8+vbio2PZ72nh6c1ucWmMdFqbc9tBc6C3zjBE7+oDXJ2fv7+4PJs6x7DPp48nlzfnVx8/nH28sXu0vmvA0N0T1cYjB5hwaYRiPcCuzEvU+na4tW++1rAm857m4dI2EanBw+QOBrorN+K5aPfp57cnYp3Ubgcm8yjyrTFHyu/0+vPow/WNmAdxPAvCe6dxFp0x8cCKi2KgWIz0Aj/rT3skzoWrFXYb7IMzmKsPtK+Bme1zTsK/42Vca0nvJJQIF91lXRElkk+Z9RVIO58FLVKyX6d43ELGc5BNQb4A4Xt4eP+Av7oZLm5nDtb1AsS9XDIw6QWnXYGiRYn2hAqBoHaa8Qk8iRdh7QRQKWNQlx40hmU5Htu5vTY1CnfC3uit+jRCP3v028nZtC5dmMfrYml5Ga3YdrHOKM/6LLjagGoCG8DjbwLp0y00aBlaWpSb/brX241RgEYT+824iCYjGGUICuf/wr8tCjKVP7rHbS/QJhwfgwq+/yICqcBnJu0EU+PobOZTAhlQz18+gq4xnAKL7BDsPwX+4Y7bqm4EJR7TntXWmQUE3lK9/an6vp1aE2lHEl/N9m4sTpNCCxr7QdcGFN6OvnQFRYcAYEAL8TTyL+IUpBf6JdpkccEj26QKfERTOvWdXbDFfgDVax2xqca5FU9rORRXnp0Hgn07aSFjZpD93nA2TSNbIgud00+bIZa6uT1Mx236IjVg7shcaf2rz5nFQeKaLbUmss2xIVYytacbtYjpjd/TmTFYMcFheNyvbJCpqaRCriZBMjVEyNVYOzww4I1GLZQOQTXLqaaYMINVpk5Bcs9Hued0XCYLigA86yLanq1btKUExAYwapWruUadT/erc016Uw3MWmKysVvddjdABNfygPBN1BygWG24p9ZHy1YN7YMWEDX0gIKmNTCGFQVOaxLoQ2q1Whv6WMqGomSdg6CnOE4+tXw67tH4KPNtLT00+RmOYz5vowEjQlmJ5ZnyjH4iAtEkQJ1qrdUhTHCAynUxdeZgvAAnDXnGdO48kjKiL5TQaPP5/hN4stTUKPJBJc62KEqk2Dr7NxKWcYvQuGE52MkNh+kaWY6Swcd2SuhFSZ7/jxxilQ8MQIyupuigsm88faTDPw1bicL94VJr3T+cHFTzdloQbDA6xD7n6YqpMwI/nEpAkPPyjdSFvEFYVkqIV+cVHYM+kAP3wUJqx/gPowj9mibSycNxq0pVP8bKxoSiLsO6gNWHc26UfPBW2WtEoeus1BcqBTEj8bsxyDII/g8Io2LyvVnmBmZjNYODxQBTWmnGrcf/9Oq+veZunIOUAriibOZfVLhTuE2nUReEaMhW9TKgkop1XHpfVeYgBCvYopPeAXgTDw1oD/bc1t5vLmWEuqC7o42oXtxZyKMghq6E9v5NZedYqlSBiu/rfKGgIQeBMctYjb+49t+enb8HxnvLlTZga6tNy4ybU21qDHJswx73oYYz2rpksE3MRA751vubacPql+GUJoImdqTItbV1VatqVNCoEWqyTJ1RR742bmGPLBgOfYuMx2P/hHDuY7H2eKTJwgzVSmuk/9axSFPdRAqlwmxTDDiO8zcpMxGIYgUaQ2DUGXUvuc8MDhHMMa9xdfVhhEvB+vWzgFVQUCE/VumEwRo0exWOaickdtBYo7YNgwhkBRbuc9SEY/8UIjIhmz7ctys4GyRsFU7uJDUmdRqtsaGxKnVwsV1pcbir2GJoLH1tHWvi7FkEs0ZB5oLQmmGl30RXJltFyMDXfRMHfyzkdnr14cPFp5vh3kqPb4i5sWYio9XXKsCaAp7IYiGjxg5VEBAL0iO0BG2q0jkBC1msYZGMLtPyHHRlZAprdOoZp82xHawsnm7MLLZ6aSWr8NOYuI23G8yFVhG5mWiVnWMq0qp7sAdhxsvHSIiuC+Sw7j4zofE2yERsIjWfY9U9P2wBg6GgQ4IDZxIutSLSOjbiCOafXaM/ade+mwih2XbQxZF9PSp+Nynfxhsm/Z5BPz/iWHqab7+tNkALPcr3kUmOeSWW87CXeZBDySbLfWNdgHHrCT1W8iKQqvjDZbx11K9jHDMfM+kK52mGuiUfDDN+LY+qk/B51sPq7spRMXyOVdTurs65YQUv3gufcmGlpkrW6ZoicvWTLkoeRjIDIxpZuX7OhdRgXnKJ307OqHCkVBSEBfVB7ogn3gV59AAoAAelJOI372dgg4Ksl3CLT8JAEyUS8/xcIldKQrTn7En29KhnRmWPJ2el5/pL3hpBjaFoFY7apbW4Q9uFwwBFQhPJLJg2U0/fegOLc7QqmPQEVufO9fuTywm9vXs0G99+10zbfXf3VBicUkz81+vPonrd9Xy4lY910N756vJMXJ/c3Ezsnes9J97r+ZMunB4+czgdtmcKrV8OVjS244xW0Q5a593KfV0KTirRJ2Hj+5ZmJIfiiNN/ZPEf4SynTmjwAMBZw/4f1CFJIhUKj7Hgk/ncJ3cMA6w13VSKrC6kAT043Zlh6roVNfndOnxgE/6gEGbP7qYCTWvPTnjNUkRuX9iuY8sOGkrKJlJrJKUctCTllIMxgSIHc4v4gqQefuuA9i6wXA5GHr+4/j0lkYU+PVXDW/kEJiSVbNKQY4/1OWEnDCp29r4d31nPmui5Un8cs2OXOxZm0ECnKKqxSapXUfB7SIHpBsj0DJOsJXVMRv6OnA2OgJVuW9UAdziiYq1253MAtTMx+kC8Mzs4aDOouWJIasfCAiiHsqkUBPQROby4BJxCliUVptzdOjwI9VtfL0+1QvzSoBgVcv2alLWqlRPQ71ixJmDY94B11/PV/koau76hgmWr9AG3LMH/oUA+6s1YLtLmK9sgbryG7WwDBjX4WT4/F1VfzZZkjeDqIOnQvbKawMUChi79GhLc07++GY13KHpWwQc462AhTdeTTZD4AIKRRSR4L7dEffR2gcxFaBnqTzAYDaI8he/63MGzj1dOWLc3HobQfsDDeaEN1x67le2skSm8ajARk4+WOfSsmc6MD7zw/ZrVO8PKA79QXzVTHbcHIIxyEMzpingbRumUEkHquct9TirCteKtuNCIdhYV0VvKi733nZK438vvl8Fk4aEaxB+ELfpBLxR5Nl2ShgHN3oLARfDBpwEP5QPg1vgsDTQoJjwxN2qxJnrEjdUaUl73sJBvDNN81JFo7ecwfZOI13hOIwfrmFJ3ulF7JSTlUAvSvSuEPRdhmwcvQmBkGMImg5c/z6pVE9bwsE4a6dyDaGYcDGCbdfJNod4uKLvrPCRqV6pVcSqrQG6wV1O8fMlG6XRr0UZh292zALsgJYPP5Bb8ugW9O1ubWznEga0i6vRE9b640i7opKKhhwK7Pn5fJLU+d6NM7m7HVvsX6+ltxGd3bVa/q3vuhhiD7WzWirE/f6V+F6a20YaVqJp2FTam8IiKfU3FU/33rmizpYN1rs62cjq9zs6wdSaDe37+DbyIth0YWbMtOIqNBXeOesnCOs2yf93moN3LEhkyI0w7vNSlHzD4EZO7JjRj+7v2ZHrDQkzrqSbMLHOQ7qbtzdCQF9PG9HmS+VbvtZWIOtjtVb4wi9pPu/+LDKoOcO+y99sOkC2x9dRGDKvpnvEneWeWgjfzetX4t0Iaox27Av+WVCEgVwDGMl2QseIv4iOQgOIXN1VmseLNEUt+tmikmez9wSz0i66j39/DWUxEnkpb0B8zZS7eSb4gW/uaetxIcgQS4+a+H6Uh+PfWTPwnHfxAT3Fr/x4pZHpNOQXYMgBGn+6KEeh//KK4Pb7bu3L1T15Ua+8dzjEF6xwmLGr5/Lt2ohpqPZnMmfoW470zrZdv9mT896KmztVKla1nfyZKtgq+qNV6ha92jAW7YwcdqqJaGkIKKOQU87hwqf0zqyhMz0R9xIoIbj5fo0d40yiF0evSX7iyybmib4PFkLyBXZPRqP6mMUgeduKEGgFV7X8RRDZ8ArNDUb1k48dc+kSSrR1DIthvkO20FTPco0oZVSca2svXhzK5I+5sxiy5sS6kN9kkatYfg1ZEz7khiTqpOPLWWMbo+/6H4LoE7DYnQJZVWHzIMpacB5/rMX30pYmTD/4HNKQt7d5MAAA="
  }
}
# END EMBEDDED CONTEXT HELPERS
TOOLS = Path(tempfile.mkdtemp(prefix="ace_context_tools_"))
for name, payload in EMBEDDED_HELPERS.items():
    data = gzip.decompress(base64.b64decode(payload["gzip_base64"], validate=True))
    assert hashlib.sha256(data).hexdigest() == payload["sha256"], f"Helper checksum mismatch: {name}"
    (TOOLS / name).write_bytes(data)
# Explicitly load this notebook's helper, even if an earlier notebook imported one.
spec = importlib.util.spec_from_file_location("electric_full_song", TOOLS / "electric_full_song.py")
embedded_single = importlib.util.module_from_spec(spec)
spec.loader.exec_module(embedded_single)
sys.modules["electric_full_song"] = embedded_single
spec = importlib.util.spec_from_file_location("ace_context_embedded", TOOLS / "ace_context_chunks.py")
context = importlib.util.module_from_spec(spec)
spec.loader.exec_module(context)
ROOT = context.single.discover_root(Path.cwd(), globals().get("ROOT"))
if ROOT is None:
    raise RuntimeError("Checkout bridge belum ada di VM ini. Setup notebook 10 dahulu; notebook ini belum menyediakan model/GPU")

print("Upload satu file piano asli saja; tidak perlu ZIP tools")
uploaded = files.upload()
audios = [n for n in uploaded if Path(n).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg"}]
assert len(audios) == 1, "Pilih satu file piano asli"
SOURCE = TOOLS / ("piano" + Path(audios[0]).suffix.lower())
SOURCE.write_bytes(uploaded[audios[0]])
OUT = ROOT / "runs" / ("ace_context_" + uuid.uuid4().hex[:10])
plan = context.prepare(SOURCE, OUT, context=90, margin=6, fade=.58)
PLAN_PATH = OUT / "plan.json"
print("ROOT:", ROOT, "| PLAN:", PLAN_PATH)
for chunk in plan["chunks"]:
    print(chunk["index"], "input model:", chunk["model_duration"], "detik", "core:", round((chunk["core_end"]-chunk["core_start"])/32000, 3))
print("Tanpa SongMap terverifikasi: batas ini bukan downbeat atau bar yang diketahui")


## Satu probe durasi

Opsional: `90` sudah dipilih, sehingga tidak perlu diulang. `60` tersedia sebagai kontrol; `120` hanya untuk eksperimen eksplisit meskipun sudah dilaporkan berisik. Caption, seed, FP32, CFG dan langkah tetap sama. Dengarkan bagian waktu yang sama, jangan hanya melihat status generated. Jangan menjalankan beberapa probe sekaligus di T4 yang sama.


In [ ]:
from IPython.display import Audio, display

DURATION = 90  # 60 / 90 / 120
EXECUTE_PROBE = False  # True hanya pada runtime CUDA/bobot yang siap
PROBE_OUT = OUT / (f"probe_{DURATION}_" + uuid.uuid4().hex[:8])
probe_report = context.single.generate(ROOT, context.generation_source(OUT, plan), PROBE_OUT,
    start=0, duration=DURATION, execute=EXECUTE_PROBE)
print("Status:", probe_report["status"], "|", PROBE_OUT)
if EXECUTE_PROBE:
    display(Audio(filename=str(PROBE_OUT / "mix_preview.mp3"), normalize=False))


In [ ]:
# Jalankan setelah probe selesai, atau setelah gagal untuk report yang tersedia.
feedback = PROBE_OUT / "full_song_feedback.zip"
if feedback.is_file():
    files.download(str(feedback))
else:
    print("Belum ada feedback inferensi. Probe dry-run tidak membuat hasil model.")


## Chunk seluruh lagu, setelah durasi konteks dipilih

Plan awal membatasi **seluruh konteks model menjadi 90 detik termasuk margin**. Jika memilih cap lain, buat plan baru dengan `context.prepare(SOURCE, NEW_OUT, context=60, margin=6, fade=.58)` dan perbarui `PLAN_PATH`. Jangan mengganti plan yang telah memiliki hasil.

Generasi sequential dan memuat model lagi per chunk. Margin/reload menambah waktu. Chunk selesai dipakai kembali setelah verifikasi; proses diffusion yang gagal diulang sebagai attempt baru. Seed sama tidak menjamin timbre/register/groove sama.


In [ ]:
EXECUTE_CHUNKS = False
if EXECUTE_CHUNKS:
    progress = context.run_chunks(ROOT, PLAN_PATH)
    report = context.stitch_run(PLAN_PATH)
    print("Sambungan perlu didengar:", [s["index"] for s in report["seams"] if s["manual_review"]])
    display(Audio(filename=str(PLAN_PATH.parent / "stitched/mix_full.mp3"), normalize=False))
else:
    print("Tidak menjalankan inferensi atau stitching; plan tersimpan di", PLAN_PATH)


In [ ]:
# Paket kecil: mix/bass MP3, report seam dan cuplikan seam; tanpa bobot model.
stitched = PLAN_PATH.parent / "stitched"
if (stitched / "seams.json").is_file():
    bundle = PLAN_PATH.parent / "chunked_feedback.zip"
    with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(PLAN_PATH, "plan.json")
        for path in sorted(stitched.iterdir()):
            if path.suffix in {".json", ".mp3"} or path.name.startswith("seam_"):
                archive.write(path, "stitched/" + path.name)
    files.download(str(bundle))
else:
    print("Belum ada hasil stitching lengkap.")


Equal-power crossfade **tetap dapat membatalkan fase bass**. Helper mencari kandidat berdasarkan energi/korelasi, membuang tepi model, dan melaporkan perubahan level/pitch; ia tidak menjamin sambungan musikal. Birama/downbeat tidak ditebak dari BPM. Repaint dan conditioning BPM/key belum diterapkan. Lihat `ace-context-chunks.md` di tool ZIP untuk rincian.
